In [4]:
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
file_path = "/home/xzcapnai/hepattn/hepattn/src/hepattn/experiments/mu3e/logs/mu3e-test-tracking_20260227-T134032/ckpts/epoch=021-val_loss=6.14010_test_eval.h5"

events_data = {}

with h5py.File(file_path, "r") as f:
    for event_id in f.keys():
        # Extract the boolean masks
        # Shape: (queries, total_hits) -> e.g., (25, 253)
        p_mask = f[f"{event_id}/preds/final/track_hit_valid/track_hit_valid"][0]
        t_mask = f[f"{event_id}/targets/particle_hit_valid"][0]
        
        # Extract which "slots" are actually valid tracks
        p_valid = f[f"{event_id}/preds/final/track_valid/track_valid"][0]
        t_valid = f[f"{event_id}/targets/particle_valid"][0]
        
        events_data[event_id] = {
            "pred_masks": p_mask[p_valid],   # Only keep actual predicted tracks
            "true_masks": t_mask[t_valid]    # Only keep actual true particles
        }

print(f"Successfully processed {len(events_data)} events.")

In [63]:
p = events_data['21875']['pred_masks']
t = events_data['21875']['true_masks']

In [ ]:
intersection = np.logical_and(p[:, None, :], t[None, :, :]).sum(axis=-1)
union = np.logical_or(p[:, None, :], t[None, :, :]).sum(axis=-1)

iou = intersection / union

best_match_idx = np.argmax(iou, axis=1)
best_iou_vals = np.max(iou, axis=1)

for i, (match, score) in enumerate(zip(best_match_idx, best_iou_vals)):
    print(f"Pred Track {i} matched True Particle {match} with IoU: {score:.2f}")

In [ ]:
with h5py.File(file_path, "r") as f:
    # List all root datasets
    print("Num Keys in HDF5:", len(list(f.keys())))
    eval_keys = list(f.keys())
    event_0 = f[eval_keys[0]]
    
    # Access a specific batch/event
    #preds = f['preds'][:]
    #targets = f['targets'][:]
    
    #print(f"Predictions shape: {preds.shape}")

In [ ]:
def print_structure(name, obj):
    print(name, obj)

with h5py.File(file_path, 'r') as f:
    f.visititems(print_structure)

In [ ]:
with h5py.File(file_path, "r") as f:
    event_id = "21874"
    
    # Get predictions and truth for this event
    pred_masks = f[f"{event_id}/preds/final/track_hit_valid/track_hit_valid"][0] # shape = (25, 253)
    true_masks = f[f"{event_id}/targets/particle_hit_valid"][0]                  # shape = (25, 253)
    
    # Check the first predicted track
    #if pred_masks.any():
    #    first_track = pred_masks[0]
    #    print(f"Track 0 predicted with {first_track.sum()} hits.")

In [ ]:
import numpy as np

threshold = 0.75
all_efficiencies = []
all_fake_rates = []

for event_id, data in events_data.items():
    p = data['pred_masks']
    t = data['true_masks']
    
    if len(t) == 0: continue  # Skip empty events
    if len(p) == 0:
        all_efficiencies.append(0)
        continue

    # 1. Calculate IoU Matrix (Preds x Truth)
    intersection = np.logical_and(p[:, None, :], t[None, :, :]).sum(axis=-1)
    union = np.logical_or(p[:, None, :], t[None, :, :]).sum(axis=-1)
    iou = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)

    # 2. Efficiency: How many true particles were found?
    # A true particle is "found" if any prediction matches it above threshold
    found_true = np.any(iou >= threshold, axis=0)
    efficiency = found_true.sum() / len(t)
    all_efficiencies.append(efficiency)

    # 3. Fake Rate: How many predictions are garbage or duplicates?
    # A prediction is "good" if it matches a unique true particle best
    # To be strict, we count how many preds fail to meet the threshold
    good_preds = np.any(iou >= threshold, axis=1)
    fake_rate = (len(p) - good_preds.sum()) / len(p)
    all_fake_rates.append(fake_rate)

print(f"--- Test Set Results (Threshold: {threshold}) ---")
print(f"Mean Efficiency: {np.mean(all_efficiencies):.4f}")
print(f"Mean Fake Rate:  {np.mean(all_fake_rates):.4f}")